# Stim Flip Simulator

In [ ]:
import stim
import stim
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## Toy Example Data Generation

### X-Error

In [2]:
import stim

# 1. Initialize FlipSimulator
# We use batch_size=1 for a single shot.
# Setting disable_stabilizer_randomization=True is critical: it prevents the 
# simulator from injecting random flips during measurements or initialization, 
# strictly isolating the deliberate errors we want to track.
sim = stim.FlipSimulator(batch_size=2, disable_stabilizer_randomization=True)

# 2. Break the circuit down into chunks to track the error step-by-step
initial_circuit = stim.Circuit("""
    # Create a Bell pair between Q0 and Q1
    H 0
    CX 0 1
""")

error_circuit = stim.Circuit("""
    # Deliberately inject an X error on Q1 with 100% probability
    X_ERROR(1.0) 1
""")

propagation_circuit = stim.Circuit("""
    # Apply a CNOT from Q1 (control) to Q2 (target)
    CX 1 2
""")

# --- Step 1: Clean State ---
print("--- After Initial Gate Setup ---")
sim.do(initial_circuit)
print("Pauli string:", sim.peek_pauli_flips()[0]) 
# Expected: +___ (No errors on the qubits)


# --- Step 2: Injecting the Error ---
print("\n--- After Injecting X_ERROR on Q1 ---")
sim.do(error_circuit)

# Peek at the current frame for shot 0
paulis = sim.peek_pauli_flips()[0]
print("Pauli string:", paulis) # Expected: +_X_ (Error is on Q1)

# To programmatically find exactly WHERE the error is:
xs, zs = paulis.to_numpy(bit_packed=False)
x_error_locations = [q for q, has_x in enumerate(xs) if has_x]
print(f"-> Qubits suffering an X flip right now: {x_error_locations}")


# --- Step 3: Error Propagation ---
print("\n--- After CNOT from Q1 to Q2 ---")
sim.do(propagation_circuit)

paulis = sim.peek_pauli_flips()[0]
print("Pauli string:", paulis) # Expected: +_XX 

# Check where the errors are now
xs, zs = paulis.to_numpy(bit_packed=False)
x_error_locations = [q for q, has_x in enumerate(xs) if has_x]
print(f"-> Qubits suffering an X flip right now: {x_error_locations}")

--- After Initial Gate Setup ---
Pauli string: +__

--- After Injecting X_ERROR on Q1 ---
Pauli string: +_X
-> Qubits suffering an X flip right now: [1]

--- After CNOT from Q1 to Q2 ---
Pauli string: +_XX
-> Qubits suffering an X flip right now: [1, 2]


In [3]:
paulis.to_numpy(bit_packed=False)

(array([False,  True,  True]), array([False, False, False]))

### Z-Error

In [4]:
import stim

# Initialize FlipSimulator
sim = stim.FlipSimulator(batch_size=1, disable_stabilizer_randomization=True)

# --- Define the Circuit Components ---
# Qubits 0 and 1 are Data. Qubit 2 is the Auxiliary (Measure) qubit.

# A standard clean XX measurement layer
clean_xx_layer = stim.Circuit("""
    R 2         # Reset Aux qubit
    H 0 1       # Rotate Data to X basis
    CX 0 2      # Entangle Q0 to Aux
    CX 1 2      # Entangle Q1 to Aux
    H 0 1       # Rotate Data back to Z basis
    M 2         # Measure Aux
""")

# We will break down Layer 2 to watch the error propagate gate-by-gate
layer2_step1 = stim.Circuit("R 2 \n H 0 1")
layer2_step2 = stim.Circuit("CX 0 2")
layer2_step3 = stim.Circuit("CX 1 2")
layer2_step4 = stim.Circuit("H 0 1 \n M 2")

# --- Execution & Tracking ---

print("=== LAYER 1: Clean State ===")
sim.do(clean_xx_layer)
print("Pauli string after Layer 1: ", sim.peek_pauli_flips()[0])
# Expected: +___ (No errors)

print("\n=== INJECTING ERROR ===")
# Inject a Z error on Q0. 
sim.do(stim.Circuit("Z_ERROR(1.0) 0"))
print("Pauli string after Injection:", sim.peek_pauli_flips()[0])
# Expected: +Z__ (Z error sitting on Q0)

print("\n=== LAYER 2: Tracking the Flip Propagation ===")
sim.do(layer2_step1)
print("After R 2 and H 0 1:        ", sim.peek_pauli_flips()[0])
# The Z error on Q0 passes through the Hadamard (H) and becomes an X error!
# Expected: +X__

sim.do(layer2_step2)
print("After CX 0 2:               ", sim.peek_pauli_flips()[0])
# The X error on Q0 (control) copies to Q2 (target) via the CNOT.
# Expected: +X_X

sim.do(layer2_step3)
print("After CX 1 2:               ", sim.peek_pauli_flips()[0])
# Q1 has no errors, so nothing changes here.
# Expected: +X_X

sim.do(layer2_step4)
print("After H 0 1 and M 2:        ", sim.peek_pauli_flips()[0])
# The X on Q0 turns back into a Z. 
# The X on Q2 flips the measurement outcome!
# Expected: +Z_X

print("\n=== LAYER 3: Lingering Errors ===")
# Run the third layer normally. 
sim.do(clean_xx_layer)
print("Pauli string after Layer 3: ", sim.peek_pauli_flips()[0])
# Expected: +Z_X 
# (The Z error remains on Q0, and it flips the Q2 measurement AGAIN in Layer 3)

# To check if the measurement itself was flipped:
measure_flips = sim.get_measurement_flips()[0]
print(f"\nMeasurement Flips (Layer 1, Layer 2, Layer 3): {measure_flips}")
# Expected: [False, True, True]

=== LAYER 1: Clean State ===
Pauli string after Layer 1:  +___

=== INJECTING ERROR ===
Pauli string after Injection: +Z__

=== LAYER 2: Tracking the Flip Propagation ===
After R 2 and H 0 1:         +X__
After CX 0 2:                +X_X
After CX 1 2:                +X_X
After H 0 1 and M 2:         +Z_X

=== LAYER 3: Lingering Errors ===
Pauli string after Layer 3:  +Z_X

Measurement Flips (Layer 1, Layer 2, Layer 3): [False]


### XZ-Errors

In [5]:
import stim

# 1. Initialize FlipSimulator
sim = stim.FlipSimulator(batch_size=1, disable_stabilizer_randomization=True)

# 2. Define the Alternating Measurement Circuits
# Q0, Q1 = Data Qubits | Q2 = Auxiliary Qubit

xx_layer = stim.Circuit("""
    R 2         # Reset Aux
    H 0 1       # Rotate Data to X basis
    CX 0 2      # Parity check
    CX 1 2
    H 0 1       # Rotate Data back to Z basis
    M 2         # Measure Aux
""")

zz_layer = stim.Circuit("""
    R 2         # Reset Aux
    CX 0 2      # Parity check
    CX 1 2
    M 2         # Measure Aux
""")

# --- LAYER 1: Clean XX Measurement ---
print("--- Layer 1: XX Measurement (Clean State) ---")
sim.do(xx_layer)
print("Pauli frame:      ", sim.peek_pauli_flips()[0])


# --- INJECTING DUAL ERRORS ---
print("\n--- Injecting X on Q0 and Z on Q1 ---")
sim.do(stim.Circuit("""
    X_ERROR(1.0) 0
    Z_ERROR(1.0) 1
"""))
print("Pauli frame:      ", sim.peek_pauli_flips()[0]) # Expected: +XZ_


# --- LAYER 2: ZZ Measurement ---
print("\n--- Layer 2: ZZ Measurement ---")
sim.do(zz_layer)
print("Pauli frame:      ", sim.peek_pauli_flips()[0]) 
# The X error on Q0 copied to Q2, triggering a measurement flip!


# --- LAYER 3: XX Measurement ---
print("\n--- Layer 3: XX Measurement ---")
sim.do(xx_layer)
print("Pauli frame:      ", sim.peek_pauli_flips()[0])
# Hadamards turn Z on Q1 into an X, copying it to Q2 and flipping this measurement!


# --- LAYER 4: ZZ Measurement ---
print("\n--- Layer 4: ZZ Measurement ---")
sim.do(zz_layer)
print("Pauli frame:      ", sim.peek_pauli_flips()[0])
# The X on Q0 triggers the ZZ measurement again!


# --- SUMMARY OF MEASUREMENT FLIPS ---
measurement_results = sim.get_measurement_flips()[0]
print("\n" + "="*45)
print("Measurement Flips across [Layer 1, Layer 2, Layer 3, Layer 4]:")
print(measurement_results) # Expected: [False, True, True, True]

--- Layer 1: XX Measurement (Clean State) ---
Pauli frame:       +___

--- Injecting X on Q0 and Z on Q1 ---
Pauli frame:       +XZ_

--- Layer 2: ZZ Measurement ---
Pauli frame:       +XZX

--- Layer 3: XX Measurement ---
Pauli frame:       +XZX

--- Layer 4: ZZ Measurement ---
Pauli frame:       +XZX

Measurement Flips across [Layer 1, Layer 2, Layer 3, Layer 4]:
[False]


### Circuit Level Noise Model

In [6]:
import stim

# Define our error probability (e.g., 2% chance of error at any point)
p = 0.02

# --- 1. Define Noisy Layers ---
# Qubits 0,1 (Data) and 2 (Aux)
# Qubits 3,4 (Data) and 5 (Aux)

noisy_xx_layer = stim.Circuit(f"""
    # Initialize with X error
    R 2 5
    X_ERROR({p}) 2 5
    
    # Basis change with 1-qubit depolarizing noise
    H 0 1 3 4
    DEPOLARIZE1({p}) 0 1 3 4
    
    # First CNOT step with 2-qubit depolarizing noise
    CX 0 2 3 5
    DEPOLARIZE2({p}) 0 2 3 5
    
    # Second CNOT step with 2-qubit depolarizing noise
    CX 1 2 4 5
    DEPOLARIZE2({p}) 1 2 4 5
    
    # Revert basis change with 1-qubit depolarizing noise
    H 0 1 3 4
    DEPOLARIZE1({p}) 0 1 3 4
    
    # Measurement is preceded by an X error
    X_ERROR({p}) 2 5
    M 2 5
""")

noisy_zz_layer = stim.Circuit(f"""
    # Initialize with X error
    R 2 5
    X_ERROR({p}) 2 5
    
    # CNOTs with 2-qubit depolarizing noise
    CX 0 2 3 5
    DEPOLARIZE2({p}) 0 2 3 5
    
    CX 1 2 4 5
    DEPOLARIZE2({p}) 1 2 4 5
    
    # Measurement is preceded by an X error
    X_ERROR({p}) 2 5
    M 2 5
""")

# --- 2. Build the Full Schedule ---
full_schedule = (
    noisy_xx_layer + 
    noisy_xx_layer + 
    noisy_zz_layer + 
    noisy_zz_layer + 
    noisy_xx_layer
)

# --- 3. Execute and Track with FlipSimulator ---
# Notice we DO NOT disable randomization here. We want Stim to roll the 
# dice based on our 'p' values and naturally inject the circuit-level noise.
sim = stim.FlipSimulator(batch_size=1)

print("=== Simulating 5 Layers of Circuit-Level Noise ===\n")

layers = [
    ("Layer 1 (XX)", noisy_xx_layer),
    ("Layer 2 (XX)", noisy_xx_layer),
    ("Layer 3 (ZZ)", noisy_zz_layer),
    ("Layer 4 (ZZ)", noisy_zz_layer),
    ("Layer 5 (XX)", noisy_xx_layer),
]

for name, layer_circuit in layers:
    sim.do(layer_circuit)
    
    # Peek at what errors the noise model randomly injected and propagated
    current_paulis = sim.peek_pauli_flips()[0]
    
    print(f"--- {name} ---")
    print(f"Current Pauli Frame: {current_paulis}")
    
    # Find exact locations of any X or Z errors on the data qubits (0, 1, 3, 4)
    xs, zs = current_paulis.to_numpy(bit_packed=False)
    x_locs = [q for q, has_x in enumerate(xs) if has_x and q in [0, 1, 3, 4]]
    z_locs = [q for q, has_z in enumerate(zs) if has_z and q in [0, 1, 3, 4]]
    
    if x_locs or z_locs:
        print(f"-> Active Data Errors: X on {x_locs}, Z on {z_locs}")
    else:
        print("-> Active Data Errors: None (Clean)")
    print()

# --- 4. Review Measurement Flips ---
# Because of the noise, some measurements will be flipped!
measurement_flips = sim.get_measurement_flips()
print("=== Final Measurement Flip Record ===")
print("Note: 2 measurements per layer (Block 1 Aux, Block 2 Aux)")

# Group the 10 measurements into the 5 layers
for i in range(5):
    m1 = measurement_flips[i*2]
    m2 = measurement_flips[i*2 + 1]
    # print(f"Layer {i+1}: Aux Q2 Flipped? {m1:<5} | Aux Q5 Flipped? {m2}")
    print(m1, " - ", m2)

=== Simulating 5 Layers of Circuit-Level Noise ===

--- Layer 1 (XX) ---
Current Pauli Frame: +__ZZZZ
-> Active Data Errors: X on [], Z on [3, 4]

--- Layer 2 (XX) ---
Current Pauli Frame: +___YYZ
-> Active Data Errors: X on [3, 4], Z on [3, 4]

--- Layer 3 (ZZ) ---
Current Pauli Frame: +ZZ_YY_
-> Active Data Errors: X on [3, 4], Z on [0, 1, 3, 4]

--- Layer 4 (ZZ) ---
Current Pauli Frame: +__ZYYZ
-> Active Data Errors: X on [3, 4], Z on [3, 4]

--- Layer 5 (XX) ---
Current Pauli Frame: +___ZZZ
-> Active Data Errors: X on [], Z on [3, 4]

=== Final Measurement Flip Record ===
Note: 2 measurements per layer (Block 1 Aux, Block 2 Aux)
[False]  -  [False]
[False]  -  [False]
[False]  -  [False]
[False]  -  [False]
[False]  -  [False]


# NN Decoder

## Toy Example

### Data Generation

In [ ]:


# ==========================================
# 1. DEFINE THE 12-QUBIT CIRCUIT
# Data Qubits: 0 to 7
# Aux Qubits: 8 to 11
# ==========================================
def build_circuit(p=0.01):
    data_qubits = "0 1 2 3 4 5 6 7"
    aux_qubits = "8 9 10 11"
    
    # Helper for CNOTs connecting Data to Aux
    cnot_pairs = "0 8 1 8 2 9 3 9 4 10 5 10 6 11 7 11"

    # XX Layer (Detects Z errors)
    xx_layer = f"""
        R {aux_qubits}
        H {data_qubits}
        DEPOLARIZE1({p}) {data_qubits}
        CX {cnot_pairs}
        DEPOLARIZE2({p}) {cnot_pairs}
        H {data_qubits}
        DEPOLARIZE1({p}) {data_qubits}
        M {aux_qubits}
    """
    
    # ZZ Layer (Detects X errors)
    zz_layer = f"""
        R {aux_qubits}
        CX {cnot_pairs}
        DEPOLARIZE2({p}) {cnot_pairs}
        M {aux_qubits}
    """
    
    # Schedule: XX -> ZZ -> XX
    return stim.Circuit(xx_layer + zz_layer + xx_layer)

# ==========================================
# 2. GENERATE TRAINING DATA USING FLIP SIMULATOR
# ==========================================
def generate_dataset(num_samples, p=0.01):
    circuit = build_circuit(p)
    
    # We can batch the simulator to generate data instantly
    sim = stim.FlipSimulator(batch_size=num_samples)
    sim.do(circuit)
    
    # INPUT (X): The measurement outcomes (syndromes)
    # 3 layers * 4 aux qubits = 12 measurements per shot
    measurements = sim.peek_measurement_flips()
    X = np.array(measurements, dtype=np.float32)
    
    # LABEL (Y): The exact final errors on the 8 Data Qubits
    # We want to predict if an X or Z error is present on Q0-Q7
    final_paulis = sim.peek_pauli_flips()
    
    Y_list = []
    for shot in range(num_samples):
        xs, zs = final_paulis[shot].to_numpy(bit_packed=False)
        
        # Extract only the data qubits (0 to 7)
        x_data_errors = xs[0:8]
        z_data_errors = zs[0:8]
        
        # Concatenate into a single 16-bit array: [X0..X7, Z0..Z7]
        y_row = np.concatenate([x_data_errors, z_data_errors])
        Y_list.append(y_row)
        
    Y = np.array(Y_list, dtype=np.float32)
    
    return torch.tensor(X), torch.tensor(Y)

# ==========================================
# 3. DEFINE THE NEURAL NETWORK
# ==========================================
class SimpleQECDecoder(nn.Module):
    def __init__(self, input_size=12, output_size=16):
        super(SimpleQECDecoder, self).__init__()
        # A simple Feed-Forward Network
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_size) 
            # Note: No Sigmoid here! We use BCEWithLogitsLoss later
            # which applies the Sigmoid internally for better math stability.
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
if __name__ == "__main__":
    print("Generating Dataset...")
    # Generate 50,000 samples for training, 5,000 for testing
    X_train, Y_train = generate_dataset(50000, p=0.03) 
    X_test, Y_test = generate_dataset(5000, p=0.03)

    train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=256, shuffle=True)

    # Initialize Model, Loss, and Optimizer
    model = SimpleQECDecoder(input_size=12, output_size=16)
    
    # Because a qubit can have an X error AND a Z error independently (creating a Y error),
    # this is a "Multi-Label Classification" problem. BCEWithLogitsLoss is perfect for this.
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    print("Training Model...")
    epochs = 10
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    # ==========================================
    # 5. TEST THE DECODER
    # ==========================================
    model.eval()
    with torch.no_grad():
        test_predictions_logits = model(X_test)
        # Apply sigmoid to turn raw logits into probabilities (0.0 to 1.0)
        test_probabilities = torch.sigmoid(test_predictions_logits)
        
        # Round to 0 or 1 (did the network think an error was there?)
        predicted_errors = (test_probabilities > 0.5).float()
        
        # Calculate exactly how many shots we guessed 100% perfectly
        perfect_matches = (predicted_errors == Y_test).all(dim=1).sum().item()
        accuracy = perfect_matches / len(Y_test)
        
        print(f"\n--- Testing Results ---")
        print(f"Total Test Shots: {len(Y_test)}")
        print(f"Shots predicted 100% perfectly: {perfect_matches}")
        print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")